# SafeX FAQ Knowledge Base — Demo

This notebook demonstrates the FAQ Knowledge Base module end-to-end:
sample questions in, retrieved answers + confidence scores out.

**Before running:** make sure you've run `python scripts/ingest_data.py`
from the project root at least once, and that `.env` has a valid
`GEMINI_API_KEY`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from app.core.pipeline import get_pipeline

pipeline = get_pipeline()
print(f"Loaded {len(pipeline.retriever.faq_lookup)} FAQs.")

## Sample queries — clear, high-confidence matches

In [ ]:
sample_questions = [
    "what are your office hours",
    "how much does a website cost",
    "do you do cybersecurity",
    "whats your phone number",
    "can i talk to a real person",
]

for q in sample_questions:
    result = pipeline.answer(q, use_cache=False)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"   confidence={result['confidence']}  matched_faq_id={result['matched_faq_id']}  is_confident={result['is_confident']}")
    print()

## Sample queries — short/ambiguous (tests query rewriting)

In [ ]:
ambiguous_questions = ["pricing?", "hours", "contact"]

for q in ambiguous_questions:
    result = pipeline.answer(q, use_cache=False)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"   confidence={result['confidence']}  matched_faq_id={result['matched_faq_id']}")
    print()

## Sample query — out-of-scope (tests confidence gate / fallback)

In [ ]:
result = pipeline.answer("what's the weather like today", use_cache=False)
print(result)

## Cache demonstration

Running the same question twice — the second call should return
`cached: true` and skip the full retrieval pipeline (requires `REDIS_URL`
to be set in `.env`).

In [ ]:
import time

q = "what are your office hours"

start = time.time()
r1 = pipeline.answer(q, use_cache=True)
t1 = time.time() - start

start = time.time()
r2 = pipeline.answer(q, use_cache=True)
t2 = time.time() - start

print(f"First call:  {t1:.3f}s  cached={r1['cached']}")
print(f"Second call: {t2:.3f}s  cached={r2['cached']}")